# Desafio 1: O Paradoxo do Vazamento (Arquitetura de Pipelines)

**O que é Data Leakage (Vazamento de Dados) neste cenário:**
O Data Leakage ocorre quando informações estatísticas do conjunto de teste vazam para o conjunto de treinamento, superestimando o desempenho real do modelo. Quando o júnior aplica o `fit_transform` em todo o dataset antes do `train_test_split`, o vetorizador global TF-IDF usa os textos que deveriam ser separados para teste para calcular o peso (IDF) de todas as palavras do corpus. Na vida real, o modelo precisa lidar com vocabulários e distribuições que nunca viu.

A solução padrão da indústria é utilizar a estrutura de `Pipeline` do Scikit-Learn: o vetorizador só executa a função de aprendizado (`fit`) na amostra de treino, simulando fielmente como o sistema agirá em produção com dados inéditos.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

reviews = ["Produto ótimo", "Péssimo", "Excelente, adorei", "Quebrado e ruim", "Muito bom"]
labels = [1, 0, 1, 0, 1]

# 1. O divisor de águas: Divisão ANTES de qualquer processamento de texto
X_train, X_test, y_train, y_test = train_test_split(reviews, labels, test_size=0.4, random_state=42)

# 2. Criação do Pipeline (Industry Standard) para prevenir o Vazamento
pipeline_desafio1 = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', MultinomialNB())
])

# 3. O pipeline encapsula o .fit() do vetorizador e restringe exclusivamente aos dados X_train
pipeline_desafio1.fit(X_train, y_train)

# 4. Avaliação (O X_test passará apenas por um 'transform' isolado dentro do pipeline)
score = pipeline_desafio1.score(X_test, y_test)
print(f"Acurácia real sem vazamento: {score:.2f}")

# Desafio 2: A Caçada aos N-gramas (Contexto e Polaridade)

Modelos de linguagem clássicos que se limitam a usar unigramas (palavras isoladas) perdem a dependência de proximidade das palavras, e consequentemente, falham de maneira crítica quando encontram partículas inversoras de polaridade, como "não gostei".

Ao habilitar o parâmetro `ngram_range=(1,2)` no vetorizador, orientamos o algoritmo a contabilizar tanto palavras unitárias quanto blocos de duas palavras consecutivas (bigramas). Como resultado, o classificador mapeia o bigrama "não gostei" de forma individual no espaço matemático e aprende que sua probabilidade e ligação são fortes com a classe de sentimento Negativa.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

textos = ["O produto é bom", "Não é bom", "Gostei muito", "Não gostei do material"]
y = [1, 0, 1, 0]

# 1. Instanciando Pipeline com suporte a N-gramas (1 a 2 palavras)
pipeline_desafio2 = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1,2))),
    ('clf', MultinomialNB())
])
pipeline_desafio2.fit(textos, y)

# 2. Predição para a frase de teste traiçoeira
frase_teste = ["Não gostei, mas dizem que é bom"]
predicao = pipeline_desafio2.predict(frase_teste)

print(f"Frase a testar: '{frase_teste[0]}'")
print(f"Predição Capturada: {'Positivo' if predicao[0] == 1 else 'Negativo'}")

# 3. Interpretando e Extraindo a Importância dos n-gramas capturados (Log Probs)
tfidf = pipeline_desafio2.named_steps['tfidf']
clf = pipeline_desafio2.named_steps['clf']
features = tfidf.get_feature_names_out()

print("\n--- Features Preditivas Mais Fortes para a Classe Negativa (0) ---")
top_indices_negativos = clf.feature_log_prob_[0].argsort()[-5:][::-1]
for idx in top_indices_negativos:
    print(f"Feature Semântica: '{features[idx]}' -> (Log Probabilidade: {clf.feature_log_prob_[0][idx]:.4f})")

# Desafio 3: A Maldição do E-commerce Real (Dados Desbalanceados)

Em conjuntos de dados de avaliações de E-commerce, deparar-se com desbalanceamentos drásticos (como 95% positivo e 5% negativo) é quase uma regra. Se realizarmos um `train_test_split` puramente aleatório, há um alto risco de selecionarmos a esmagadora maioria dos elementos negativos exclusivamente para a área de teste (impedindo o modelo de aprender o que é 'negativo' durante o treino) ou exclusivamente para o treino (fazendo com que nossa avaliação final ignore o aspecto minoritário e seja superestimada).

Ao adicionarmos o parâmetro `stratify=y` no scikit-learn, garantimos uma divisão demograficamente rígida sob o capô. A função fará com que a mesma proporção inicial (no nosso caso, de avaliações Positivas contra as Negativas) seja integral e exatamente replicada nas partições de treino e teste. Dessa forma, a validação é robusta e condiz com o ambiente real.

*Nota de Engenharia*: O `train_test_split` falha quando a classe minoritária possui apenas 1 elemento (como fornecido originalmente pelo desafio), pois a divisão matematicamente quebra. No código abaixo, simulamos o reparo duplicando os dados do Toy Dataset.

In [ ]:
from sklearn.model_selection import train_test_split
from collections import Counter

X_desbalanceado = ["Amei", "Perfeito", "Excelente", "Muito bom", "Ótimo produto", 
                   "Recomendo", "Chegou rápido", "Nota 10", "Qualidade boa", "Péssimo e estragado"]
y_desbalanceado = [1, 1, 1, 1, 1, 1, 1, 1, 1, 0]

# Como o scikit-learn exige pelo menos 2 ocorrências da classe minoritária para estratificar,
# estamos duplicando a amostra para viabilizar e explicitar o impacto da demonstração.
X_ampliado = X_desbalanceado * 2
y_ampliado = y_desbalanceado * 2

# Divisão Aleatória Padrão (Vulnerável)
X_train_ruim, X_test_ruim, y_train_ruim, y_test_ruim = train_test_split(
    X_ampliado, y_ampliado, test_size=0.25, random_state=42
)

# Divisão Estratificada (Arquitetura Resiliente)
X_train_strat, X_test_strat, y_train_strat, y_test_strat = train_test_split(
    X_ampliado, y_ampliado, test_size=0.25, random_state=42, stratify=y_ampliado
)

print("================ ANÁLISE DE ESTRATIFICAÇÃO ================")
print(f"Distribuição da Base Completa  : {dict(Counter(y_ampliado))}")

print("\n[X] COMPORTAMENTO SEM STRATIFY")
print(f" - Distribuição em Treino : {dict(Counter(y_train_ruim))}")
print(f" - Distribuição em Teste  : {dict(Counter(y_test_ruim))}  <- Notem o enviesamento aleatório")

print("\n[V] COMPORTAMENTO COM STRATIFY")
print(f" - Distribuição em Treino : {dict(Counter(y_train_strat))}")
print(f" - Distribuição em Teste  : {dict(Counter(y_test_strat))}  <- Respeitando precisamente a proporção global")